# Robomimic Failure Recognition Baseline
This script demonstrates training the `FailureRecognizer` model on trajectories generated
from the robomimic simulation data via the `a2l-pr` perturbation pipeline.

In [1]:
import os
import sys
import h5py
import torch
import numpy as np

# Ensure the a2l-pr package is in the path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from a2l_pr.adapters.robomimic import RobomimicAdapter  
from a2l_pr.perturbations.generator import PerturbationGenerator, PerturbationType
from a2l_pr.learning import FailureRecoveryDataset, FailureRecoveryTrainer
from a2l_pr.models import FailureRecognizer
from a2l_pr.config.config import ConfigManager

## 1. Load Robomimic Trajectory
We will load a trajectory from the Robomimic `image.hdf5` dataset, apply a perturbation, and format
the images for our baseline model.

In [3]:
def load_robomimic_trajectory(hdf5_path, demo_key="demo_0"):
    """Loads a single trajectory dictionary from a standard robomimic HDF5 file."""
    if not os.path.exists(hdf5_path):
        print(f"HDF5 file not found: {hdf5_path}")
        return None
        
    f = h5py.File(hdf5_path, 'r')
    if "data" not in f or demo_key not in f["data"]:
        print(f"Invalid HDF5 structure or missing {demo_key}")
        f.close()
        return None
        
    demo_grp = f["data"][demo_key]
    obs_grp = demo_grp["obs"]
    
    # Reconstruct the dictionary
    trajectory = {
        'actions': demo_grp["actions"][:],
        'observations': {}
    }
    
    for key in obs_grp.keys():
        trajectory['observations'][key] = obs_grp[key][:]
        
    f.close()
    return trajectory

robomimic_hdf5_path = os.path.abspath(os.path.join(
    os.getcwd(), "..", "..", "robomimic", "robomimic", "datasets", "square", "ph", "low_dim_v15.hdf5"
))

# Try loading the trajectory
raw_trajectory = load_robomimic_trajectory(robomimic_hdf5_path)

if raw_trajectory is None:
    print("Loading a synthetic dummy trajectory for demonstration purposes.")
    traj_len = 100
    t = np.linspace(0, 1, traj_len)
    raw_trajectory = {
        'observations': {
            'robot0_eef_pos': np.column_stack((t, t, t)),
            'robot0_gripper_qpos': np.concatenate((np.zeros((50, 1)), np.ones((50, 1)))),
            # Synthesize fake images if not present
            'agentview_image': np.random.randint(0, 255, (traj_len, 84, 84, 3), dtype=np.uint8)
        },
        'actions': np.random.randn(traj_len, 7)
    }
else:
    print(f"Successfully loaded {robomimic_hdf5_path}")

# Apply Adapter
adapter = RobomimicAdapter()
trajectory = adapter.load(raw_trajectory)

# Inject config defaults
config_path = os.path.abspath(os.path.join(os.getcwd(), "..", "perturbation_config.yaml"))
config_mgr = ConfigManager.from_yaml(config_path)
dataset_defaults = config_mgr.get_dataset_defaults(trajectory['metadata']['dataset_type'])
trajectory['metadata'].update(dataset_defaults)
trajectory['metadata'].update(config_mgr.get_perturbation_config())

Successfully loaded /home/griffing52/vail/bot2bot/bot2bot/a2l/robomimic/robomimic/datasets/square/ph/low_dim_v15.hdf5


## 2. Apply Perturbations and Format Data for Learning
We'll create a small dataset containing the "original" (no failure) trajectory,
and a few "perturbed" (failure) trajectories.
We will extract the last 3 frames and the last 2 actions at the end of the trajectory.

In [4]:
generator = PerturbationGenerator()

perturbation_types = [
    PerturbationType.UNDERREACH_IDLE,
    PerturbationType.PREMATURE_CLOSE,
    PerturbationType.LATERAL_DRIFT
]

data_records = []

def extract_learning_sample(traj_dict, failure_type_id, fsm_id=0, rec_params=None):
    """Extracts the last 3 frames and 2 actions to match the dataset expectation."""
    if rec_params is None:
        rec_params = np.zeros(7)
        
    actions = traj_dict['actions']
    obs = traj_dict['observations']
    
    # We need images. Defaulting to 'agentview_image' if available.
    if 'agentview_image' in obs:
        imgs = obs['agentview_image']
    else:
        # Fallback to random if not found
        imgs = np.random.randint(0, 255, (len(actions), 84, 84, 3), dtype=np.uint8)
        
    # Get last 3 frames
    num_frames = 3
    if len(imgs) >= num_frames:
        last_frames = imgs[-num_frames:]
    else:
        last_frames = np.pad(imgs, ((num_frames - len(imgs), 0), (0,0), (0,0), (0,0)), mode='edge')
        
    # Convert from (N, H, W, C) to (N, C, H, W) and normalize to [0, 1]
    last_frames = np.transpose(last_frames, (0, 3, 1, 2)).astype(np.float32) / 255.0
    
    # Get last 2 actions
    num_act = 2
    if len(actions) >= num_act:
        last_actions = actions[-num_act:]
    else:
        last_actions = np.pad(actions, ((num_act - len(actions), 0), (0,0)), mode='edge')
        
    return {
        'frames': last_frames,
        'actions': last_actions.astype(np.float32),
        'failure_type_id': failure_type_id,
        'fsm_id': fsm_id,
        'recovery_params': rec_params.astype(np.float32)
    }

# 1. Add Original (No Failure)
print("Processing Original Trajectory (Label 0: No Failure)...")
data_records.append(extract_learning_sample(trajectory, failure_type_id=0))

# 2. Add Perturbations
for idx, p_type in enumerate(perturbation_types, start=1):
    print(f"Applying {p_type.name} (Label {idx})...")
    result = generator.apply_perturbation(trajectory, p_type, severity=0.5, seed=42)
    if result is not None:
        # We dummy the recovery FSM and params for this example based on idx
        data_records.append(extract_learning_sample(
            result.perturbed_trajectory, 
            failure_type_id=idx,
            fsm_id=min(idx, 2), # Dummy FSM id
            rec_params=np.random.randn(7) # Dummy recovery params
        ))
    else:
        print(f"Skipped {p_type.name} (not applicable).")

print(f"Created a dataset with {len(data_records)} samples.")

Processing Original Trajectory (Label 0: No Failure)...
Applying UNDERREACH_IDLE (Label 1)...
Applying PREMATURE_CLOSE (Label 2)...
Applying LATERAL_DRIFT (Label 3)...
Created a dataset with 4 samples.


## 3. Train Baseline FailureRecognizer Model
Here we instantiate our baseline model and train it for a few epochs on the generated data.

In [5]:
train_dataset = FailureRecoveryDataset(data_records)

# Since we have small images (e.g., 84x84) from robomimic, resnet18 handles it well
model = FailureRecognizer(
    num_frames=3,
    action_dim=7,
    num_action_history=2,
    num_failure_types=len(perturbation_types) + 1,
    recovery_param_dim=7,
    num_fsm_templates=3,
    vision_encoder_type="resnet18",
    hidden_dim=256
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Initializing Trainer on {device}...")

trainer = FailureRecoveryTrainer(
    model=model,
    train_dataset=train_dataset,
    batch_size=2, # Small batch size for demonstration
    lr=1e-3,
    device=device
)

print("Starting Training (Overfitting to the small batch)...")
for epoch in range(1, 11):
    loss = trainer.train_epoch()
    print(f"Epoch {epoch} Loss: {loss:.4f}")

/home/griffing52/miniconda3/envs/a2l/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/griffing52/miniconda3/envs/a2l/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Initializing Trainer on cuda...
Starting Training (Overfitting to the small batch)...


Training: 100%|██████████| 2/2 [00:00<00:00,  7.40it/s]


Epoch 1 Loss: 3.3792


Training: 100%|██████████| 2/2 [00:00<00:00, 188.39it/s]


...

Epoch 8 Loss: 1.0328


Training: 100%|██████████| 2/2 [00:00<00:00, 217.96it/s]


Epoch 9 Loss: 0.9994


Training: 100%|██████████| 2/2 [00:00<00:00, 216.60it/s]

Epoch 10 Loss: 0.6701


## 4. Inference on Original vs Perturbed Trajectories
We test the model's ability to distinguish 'no failure' from a specific failure,
using the exact inputs we processed from Robomimic.

In [6]:
model.eval()

print("\n--- INFERENCE TEST ---")

vocab_failures = {0: "no failure", 1: "underreach", 2: "premature close", 3: "lateral drift"}
vocab_fsm = {0: "idle", 1: "move-back-and-retry", 2: "re-grasp"}

with torch.no_grad():
    for i in range(len(train_dataset)):
        sample = train_dataset[i]
        frames = sample['frames'].unsqueeze(0).to(device)
        actions = sample['actions'].unsqueeze(0).to(device)
        
        outputs = model(frames, actions)
        
        pred_failure_idx = torch.argmax(outputs['failure_logits'], dim=1).item()
        pred_fsm_id = torch.argmax(outputs['fsm_logits'], dim=1).item()
        pred_rec_params = outputs['recovery_params'][0].cpu().numpy()
        
        true_failure = sample['failure_type'].item()
        
        print(f"\nSample {i+1} (True Label: {vocab_failures.get(true_failure, true_failure)})")
        
        text = model.generate_recovery_text(
            pred_failure_idx, pred_fsm_id, pred_rec_params, 
            failure_vocab=vocab_failures, fsm_vocab=vocab_fsm
        )
        print(f"Model Output: {text}")


--- INFERENCE TEST ---

Sample 1 (True Label: no failure)
Model Output: Detected 'premature close'. Executing 're-grasp' recovery with params [-0.08, 0.27, 0.21, 0.12, -0.29, 0.21, 0.45].

Sample 2 (True Label: underreach)
Model Output: Detected 'underreach'. Executing 're-grasp' recovery with params [0.05, -0.09, 0.27, -0.34, -0.43, 0.34, 0.72].

Sample 3 (True Label: premature close)
Model Output: Detected 'premature close'. Executing 're-grasp' recovery with params [0.03, 0.51, 0.06, 0.39, -0.23, -0.04, 0.35].

Sample 4 (True Label: lateral drift)
Model Output: Detected 'lateral drift'. Executing 're-grasp' recovery with params [0.68, 0.05, -0.34, -0.33, -0.03, -0.54, 0.67].


In [7]:
print("\nDone! The baseline successfully integrates with the robomimic pipeline.")


Done! The baseline successfully integrates with the robomimic pipeline.
